# 04 — Direct Mode: Signal Acquisition & I/O

This notebook covers the extended I/O capabilities of the CompactStat in direct mode:

- **High-speed traces** — acquire up to 256 current/potential samples at rates from 10 µs to 20 ms
- **External DAC/ADC ports** — output analog voltages and read external sensors
- **Digital I/O** — control digital output lines and read input lines via bitmasks
- **AC signal control** — set frequency and amplitude for impedance experiments
- **MUX channel** — switch multiplexer channels

### Prerequisites
- Driver open, device connected (see `02_device_and_instance_management.ipynb`)
- Cell configuration set (see `03_direct_mode_basics.ipynb`)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
Pyvium.set_connection_mode(1)  # EStat 4EL
Pyvium.set_current_range(2)    # 100 µA
Pyvium.set_filter(2)           # 10 kHz — needed for fast traces
print("Ready")

## 1. High-Speed Current Trace

`get_current_trace(n_points, interval)` captures a burst of current samples.

- `n_points`: number of samples (max 256)
- `interval`: time between samples in seconds (10 µs to 20 ms)

Returns a list of current values in Amperes.

> **HiSpeed mode note:** When `interval` is below 2 ms, IviumSoft automatically enters **HiSpeed mode** — the data is stored in the potentiostat's internal buffer during acquisition and transferred to the PC only after the burst completes. In this mode the maximum buffer size is **8192 points** per acquisition (32 768 for CV). Attempting more than 8192 points in a single HiSpeed burst will be truncated. For intervals ≥ 2 ms, data is streamed in real time with no buffer limit.

In [ ]:
N_POINTS = 256
INTERVAL = 0.001  # 1 ms between samples → 256 ms total burst

Pyvium.set_potential(0.1)
Pyvium.set_cell_on()
time.sleep(0.05)  # settle

current_trace = Pyvium.get_current_trace(N_POINTS, INTERVAL)
print(f"Acquired {len(current_trace)} samples")
print(f"Mean current: {np.mean(current_trace):.3e} A")
print(f"Std dev     : {np.std(current_trace):.3e} A")

times = [i * INTERVAL * 1000 for i in range(len(current_trace))]  # ms
currents_uA = [c * 1e6 for c in current_trace]  # µA

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times, currents_uA, lw=0.8)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Current (µA)")
ax.set_title(f"Current Trace — {N_POINTS} points at {INTERVAL*1000:.1f} ms intervals")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. High-Speed Potential Trace

Same API as the current trace but reads potential. Useful for checking the cell response under a current step.

In [ ]:
potential_trace = Pyvium.get_potential_trace(N_POINTS, INTERVAL)

times_ms   = [i * INTERVAL * 1000 for i in range(len(potential_trace))]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_ms, potential_trace, 'r-', lw=0.8)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Potential (V)")
ax.set_title("Potential Trace")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Pyvium.set_cell_off()

## 3. Comparing Current and Potential Traces

Acquire both signals simultaneously (two separate bursts at the same conditions) for a complete picture.

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.05)

I_trace = Pyvium.get_current_trace(128, 0.0005)   # 128 pts, 0.5 ms interval
E_trace = Pyvium.get_potential_trace(128, 0.0005)

Pyvium.set_cell_off()

t_ms = [i * 0.5 for i in range(128)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(t_ms, [c * 1e6 for c in I_trace], 'b-', lw=0.8)
ax1.set_ylabel("Current (µA)")
ax1.grid(True, alpha=0.3)

ax2.plot(t_ms, E_trace, 'r-', lw=0.8)
ax2.set_xlabel("Time (ms)")
ax2.set_ylabel("Potential (V)")
ax2.grid(True, alpha=0.3)

fig.suptitle("Current and Potential Traces")
plt.tight_layout()
plt.show()

## 4. External DAC Output

The CompactStat has two external DAC channels (0 and 1) on the external port. Use them to output analog control signals to external equipment (e.g., trigger a lamp, control a valve, sync with another instrument).

Output range: typically ±10 V (check your device specification).

In [ ]:
# Set DAC channel 0 to 1.5 V
Pyvium.set_dac(0, 1.5)
print("DAC channel 0 set to 1.5 V")

# Set DAC channel 1 to -0.5 V
Pyvium.set_dac(1, -0.5)
print("DAC channel 1 set to -0.5 V")

# Reset both to 0 V when done
Pyvium.set_dac(0, 0.0)
Pyvium.set_dac(1, 0.0)
print("DAC channels reset to 0 V")

## 5. External ADC Input

`get_adc(channel)` reads voltage on one of the 8 external ADC input channels (0–7). Useful for reading external sensors (temperature, pH, flow rate, etc.) in synchrony with an electrochemical measurement.

In [ ]:
for channel in range(4):  # read channels 0–3
    voltage = Pyvium.get_adc(channel)
    print(f"ADC channel {channel}: {voltage:.4f} V")

## 6. Digital Output

`set_digital_output(bitmask)` controls the digital output lines. Each bit in the integer maps to one digital output line.

```
bitmask = 0b00000101  →  lines 0 and 2 HIGH, others LOW
```

In [ ]:
# Set digital output line 0 HIGH
Pyvium.set_digital_output(0b00000001)
print(f"Digital out: 0b{0b00000001:08b}  (line 0 HIGH)")
time.sleep(0.5)

# Set lines 0 and 2 HIGH
Pyvium.set_digital_output(0b00000101)
print(f"Digital out: 0b{0b00000101:08b}  (lines 0 and 2 HIGH)")
time.sleep(0.5)

# All lines LOW
Pyvium.set_digital_output(0b00000000)
print("Digital out: all lines LOW")

## 7. Digital Input

`get_digital_input()` returns a bitmask of the current state of the digital input lines.

In [ ]:
digin = Pyvium.get_digital_input()
print(f"Digital input bitmask : {digin} (0b{digin:08b})")

# Decode individual lines
for line in range(8):
    state = "HIGH" if digin & (1 << line) else "LOW"
    print(f"  Line {line}: {state}")

## 8. AC Signal Control (for Manual EIS)

`set_ac_frequency()` and `set_ac_amplitude()` configure the AC perturbation signal applied to the cell. In direct mode this lets you manually apply an AC signal at a fixed frequency — useful for single-frequency impedance measurements or verifying setup before running a full EIS method.

For automated EIS sweeps, use `start_method()` with an EIS method file instead.

In [ ]:
# Set AC perturbation: 10 mV amplitude at 1 kHz
Pyvium.set_ac_frequency(1000.0)   # Hz
Pyvium.set_ac_amplitude(0.010)    # V (10 mV)
print("AC signal: 10 mV at 1 kHz")

Pyvium.set_potential(0.0)         # DC bias
Pyvium.set_cell_on()
time.sleep(0.5)

# Read resulting current for a rough impedance estimate
I = Pyvium.get_current()
E = Pyvium.get_potential()
print(f"DC potential: {E:.4f} V | DC current: {I:.3e} A")

Pyvium.set_cell_off()

## 9. MUX Channel Selection

`set_mux_channel(n)` switches the multiplexer to the given channel (0-indexed). Use this when a multiplexer accessory is connected to rotate through multiple working electrodes.

In [ ]:
NUM_MUX_CHANNELS = 4
DWELL_TIME = 1.0  # seconds per channel

Pyvium.set_potential(0.0)
Pyvium.set_cell_on()

results = []
for ch in range(NUM_MUX_CHANNELS):
    Pyvium.set_mux_channel(ch)
    time.sleep(DWELL_TIME)
    I = Pyvium.get_current()
    results.append((ch, I))
    print(f"  MUX ch {ch}: I = {I:.3e} A")

Pyvium.set_cell_off()
Pyvium.set_mux_channel(0)  # reset to channel 0

## Cleanup

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("Done")

---

## Summary

| Capability | Method | Key constraints |
|-----------|--------|----------------|
| Current burst | `get_current_trace(n, interval)` | n ≤ 256, interval 10 µs–20 ms |
| Potential burst | `get_potential_trace(n, interval)` | same |
| WE2 current burst | `get_current_we2_trace(n, interval)` | BiStat required |
| Analog output | `set_dac(channel, volts)` | channels 0 or 1 |
| Analog input | `get_adc(channel)` | channels 0–7 |
| Digital output | `set_digital_output(bitmask)` | integer bitmask |
| Digital input | `get_digital_input()` | returns bitmask |
| AC frequency | `set_ac_frequency(hz)` | float in Hz |
| AC amplitude | `set_ac_amplitude(volts)` | float in V |
| MUX channel | `set_mux_channel(n)` | 0-indexed |

## Next

- **`05_bipotentiostat_and_we32.ipynb`** — BiStat WE2 channel and 32-channel WE32 array
- **`06_method_mode.ipynb`** — Full electrochemical method execution (LSV, CV, EIS)